# Lost in Summary: When Do LLM Review Summaries Drop Complaints?

CS 6120 Final Project, Fall 2026

**Data:** Amazon Reviews '23 (Hou et al., 2024), https://amazon-reviews-2023.github.io/

Reviews are **streamed** from Hugging Face, so the full dataset is never downloaded. Each sample is saved to `data/`
the first time; re-running loads the saved file instead.

## 1. Setup

In [ ]:
%pip install -q datasets pandas pyarrow

In [ ]:
import json
from itertools import islice
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfFileSystem

# Categories where safety complaints are realistic (names must match the dataset site exactly)
CATEGORIES = ["Electronics", "Beauty_and_Personal_Care", "Home_and_Kitchen"]

SAMPLE_SIZE = 20_000   # reviews per category
SHUFFLE_BUFFER = 10_000
SEED = 42

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

HF_BASE = "hf://datasets/McAuley-Lab/Amazon-Reviews-2023/raw"
REVIEW_COLUMNS = ["rating", "title", "text", "asin", "parent_asin", "user_id",
                  "timestamp", "helpful_vote", "verified_purchase"]

## 2. Load reviews

In [ ]:
def stream_reviews(category, n):
    """Stream n reviews from one category and return them as a DataFrame.

    The shuffle is within a 10K-review window, not the whole file, so this is
    a spread-out sample rather than a perfectly random one.

    >>> df = stream_reviews("All_Beauty", 5)
    >>> len(df)
    5
    """
    stream = load_dataset(
        "json",
        data_files=f"{HF_BASE}/review_categories/{category}.jsonl",
        split="train",
        streaming=True,
    )
    rows = stream.shuffle(buffer_size=SHUFFLE_BUFFER, seed=SEED).take(n)
    df = pd.DataFrame(list(rows))[REVIEW_COLUMNS]
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    df["category"] = category
    return df


def load_reviews(category, n=SAMPLE_SIZE):
    """Load a saved sample from data/, or stream and save it if missing."""
    path = DATA_DIR / f"reviews_{category}_{n}.parquet"
    if path.exists():
        print(f"{category}: loading saved sample from {path}")
        return pd.read_parquet(path)
    print(f"{category}: streaming {n:,} reviews...")
    df = stream_reviews(category, n)
    df.to_parquet(path, index=False)
    print(f"{category}: saved to {path}")
    return df

In [ ]:
reviews = pd.concat([load_reviews(c) for c in CATEGORIES], ignore_index=True)
print(f"\nTotal: {len(reviews):,} reviews")

### Quick look

In [ ]:
reviews.head()

In [ ]:
# Star ratings per category (complaint reviews will need matching ratings later)
reviews.groupby("category")["rating"].value_counts().unstack().astype(int)

In [ ]:
# Distinct products, and how many reviews each one has in the sample
per_product = reviews.groupby(["category", "parent_asin"]).size()
per_product.groupby("category").describe()[["count", "mean", "max"]]

In [ ]:
# Review length in words
reviews["n_words"] = reviews["text"].str.split().str.len()
reviews.groupby("category")["n_words"].describe()[["mean", "50%", "max"]]

### Product metadata (peek)

Product titles and categories are in separate metadata files. We'll need them to pool similar products
(e.g., all USB chargers), since the sample has only ~1 review per product.

In [ ]:
# Read the first few lines directly; load_dataset guesses column types wrong on the metadata files
meta_path = f"{HF_BASE}/meta_categories/meta_{CATEGORIES[0]}.jsonl".removeprefix("hf://")
with HfFileSystem().open(meta_path) as f:
    meta_peek = pd.DataFrame([json.loads(line) for line in islice(f, 5)])

meta_peek[["parent_asin", "title", "main_category", "categories", "average_rating", "rating_number"]]

## 3. Find complaint reviews

*To do:* keyword search + LLM filter for each complaint (half safety, half cosmetic), then hand-check a sample.

## 4. Build controlled review sets

*To do:* fixed-size sets (50-100 reviews) at 2%, 5%, 10%, 20%, 40% complaint prevalence; swap in complaint reviews
for non-complaint reviews with the same star rating; place them at start / middle / end / random.

## 5. Generate summaries

*To do:* run each model (Llama 3.1 8B Instruct, Qwen2.5 7B Instruct, one API model) on every set, plus the two
mitigation prompts. Long LLM runs on the Mac need a load slot first.

## 6. Check coverage (LLM judge)

*To do:* yes/no judge prompt; hand-label ~200 summaries and report Cohen's kappa.

## 7. Analysis

*To do:* inclusion rates, inclusion curves, threshold prevalence, logistic regression (statsmodels).